# True F1, a common operating point, and Bonferroni intervals (B vs N, plus all six models)

**No training. About 45-75 minutes on one T4 GPU** (most of it is the 10,000-resample mAP bootstrap; set `N_BOOT_MAP_BIG = 2000` to make it faster).

What this notebook answers (the three points the evaluation boards raised):
1. **True F1.** The thesis metric `p1_image` divides the sum of matched Dice by *M + unmatched*; a usual F1 would divide 2 x the sum by *2M + unmatched*. Here both are reported, for B and N, on the test set and on the 1200 OOF images.
2. **Same operating point.** B and N were frozen at different post-processing values (Table 3.4). Here both models are scored at the *same* values: B's values, N's values, and one joint setting chosen on the OOF images only (never on the test set).
3. **Bonferroni.** Paired N - B intervals at 95 % and at 99.67 % (= 1 - 0.05/15), with 10,000 resamples for every metric, mAP included. If `unified_test_predictions.npz` is attached, the 99.67 % intervals are also computed for all 15 pairs of the six models (Dice, official-style F1, true F1, instance F1).

**Inputs:** (1) ARCADE dataset (nirmalgaud), (2) the output of the first VG-DeepLab notebook (`dl3_r101_f0..4.pt`, `vg_dl3_f0..4.pt`), (3) optional: a dataset with `unified_test_predictions.npz` (from the unified evaluation output). GPU = T4.

**Output:** folder `truef1_outputs` + `truef1_outputs.zip`. Send the zip back.

In [1]:
# Do NOT let pip touch torch: --no-deps keeps the preinstalled torch exactly as it is.
!pip install -q --no-deps segmentation-models-pytorch
!pip install -q ultralytics
print("Installs complete.")

Installs complete.


In [2]:
import os, sys, re, json, math, time, shutil, zipfile, itertools, warnings, subprocess, traceback
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import torch, torch.nn as nn, torch.nn.functional as F
from torch.amp import autocast
for _try in range(8):          # smp needs a few small packages; install any that are missing WITHOUT touching torch
    try:
        import segmentation_models_pytorch as smp; break
    except ModuleNotFoundError as e:
        print("missing package:", e.name, "-> installing (no deps)")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", e.name.split(".")[0].replace("_", "-")])
warnings.filterwarnings("ignore")

# ============================================================
# CONFIG
# ============================================================
SMOKE_TEST = False         # True = quick plumbing check on 40 val + 40 test images (numbers are meaningless). False = the real run.
IMG_SIZE, STENOSIS_CATEGORY_ID = 512, 26
N_BOOT, N_BOOT_MAP = 10000, 200          # bootstrap resamples (means / mAP)
HEADLINE = "balanced-DEPLOY"
YOLO_CONF, YOLO_MAX_DET = 0.001, 20      # same as the YOLOv8s-seg notebook
CAND_THR, CAND_MIN_AREA, CAND_MAX = 0.30, 100, 8      # candidate list used for mAP of the semantic models (same as the earlier notebooks)
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu"); USE_CUDA = DEVICE.type == "cuda"
ON_KAGGLE = Path("/kaggle/working").exists()
BASE = Path("/kaggle/working") if ON_KAGGLE else (Path("/content") if Path("/content").exists() else Path.cwd())
OUT_DIR = BASE / "truef1_outputs"; OUT_DIR.mkdir(parents=True, exist_ok=True)
SEARCH_ROOTS = [Path(p) for p in os.environ.get("UNI_WEIGHT_ROOTS", "/kaggle/input:/content/weights:/content/drive/MyDrive/arcade_stenosis_runs").split(":")]
N_BOOT_MAP_BIG = 10000                   # mAP resamples for the Bonferroni intervals (2000 is faster)
ALPHA_BONF = 0.05 / 15                   # 15 pairs of six models
if SMOKE_TEST: N_BOOT, N_BOOT_MAP, N_BOOT_MAP_BIG = 500, 20, 50
ERRORS = []
print("Torch", torch.__version__, "| smp", smp.__version__, "|", torch.cuda.get_device_name(0) if USE_CUDA else "NO GPU (enable a GPU in the notebook settings!)", "| SMOKE_TEST =", SMOKE_TEST)

# frozen operating points of the two 5-fold ensembles (chosen on out-of-fold predictions in the first VG-DeepLab run: frozen_choices_v3.json)
FROZEN_OPS = {
    "B": {"balanced-DEPLOY": dict(thr=0.40, min_area=150, tau=0.5, rel=0.75, max_k=99, top1=False),
          "maxF1-ARCADE":    dict(thr=0.40, min_area=331, tau=0.6, rel=0.75, max_k=3,  top1=True)},
    "N": {"balanced-DEPLOY": dict(thr=0.35, min_area=150, tau=0.2, rel=0.75, max_k=3,  top1=False),
          "maxF1-ARCADE":    dict(thr=0.45, min_area=150, tau=0.7, rel=0.0,  max_k=3,  top1=True)},
}

Torch 2.11.0+cu130 | smp 0.6.0 | Tesla T4 | SMOKE_TEST = False


In [3]:
# ============================================================
# ARCADE data + instance / metric code (identical logic to the VG-DeepLab and YOLOv8s-seg notebooks)
# ============================================================
def find_arcade_root():
    for root in [Path(p) for p in os.environ.get("UNI_DATA_ROOTS", "/kaggle/input").split(":")] + [Path.home() / ".cache" / "kagglehub"]:
        if root.exists():
            c = [p for p in root.rglob("train.json") if "stenosis" in str(p).lower()]
            if c: return c[0].parent.parent.parent
    return None

DATASET_ROOT = find_arcade_root()
if DATASET_ROOT is None:
    raise RuntimeError("ARCADE dataset not found. In Kaggle: Add Input -> search 'arcade-dataset' (nirmalgaud) -> Add, then Save Version -> Save & Run All.")
for s in ["train", "val", "test"]:
    assert (DATASET_ROOT / s / "annotations" / f"{s}.json").exists() and (DATASET_ROOT / s / "images").exists()
print("ARCADE root:", DATASET_ROOT)

class Inst:
    """One instance mask, cropped to its bounding box."""
    __slots__ = ("y0", "x0", "m", "area", "score", "extra")
    def __init__(self, y0, x0, m, score=1.0, extra=None):
        self.y0, self.x0, self.m = int(y0), int(x0), m
        self.area = int(m.sum()); self.score, self.extra = float(score), extra
    @property
    def y1(self): return self.y0 + self.m.shape[0]
    @property
    def x1(self): return self.x0 + self.m.shape[1]

def inst_from_full(full_bool, score=1.0):
    ys, xs = np.nonzero(full_bool)
    if len(ys) == 0: return None
    y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
    return Inst(y0, x0, full_bool[y0:y1, x0:x1].copy(), score)

def inst_inter(a, b):
    y0, y1 = max(a.y0, b.y0), min(a.y1, b.y1); x0, x1 = max(a.x0, b.x0), min(a.x1, b.x1)
    if y0 >= y1 or x0 >= x1: return 0
    return int(np.count_nonzero(a.m[y0 - a.y0:y1 - a.y0, x0 - a.x0:x1 - a.x0] & b.m[y0 - b.y0:y1 - b.y0, x0 - b.x0:x1 - b.x0]))

def inst_dice(a, b):
    s = a.area + b.area
    return 0.0 if s == 0 else 2.0 * inst_inter(a, b) / s

def inst_iou(a, b):
    i = inst_inter(a, b); u = a.area + b.area - i
    return 0.0 if u == 0 else i / u

def box_iou(a, b):
    ih, iw = min(a.y1, b.y1) - max(a.y0, b.y0), min(a.x1, b.x1) - max(a.x0, b.x0)
    if ih <= 0 or iw <= 0: return 0.0
    i = ih * iw; u = (a.y1 - a.y0) * (a.x1 - a.x0) + (b.y1 - b.y0) * (b.x1 - b.x0) - i
    return i / u

def paint(insts, shape):
    canvas = np.zeros(shape, bool)
    for i in insts: canvas[i.y0:i.y1, i.x0:i.x1] |= i.m
    return canvas

def greedy_match(preds, gts, fn):
    pairs = []
    for i, p in enumerate(preds):
        for j, g in enumerate(gts):
            s = fn(p, g)
            if s > 0: pairs.append((s, i, j))
    pairs.sort(key=lambda t: -t[0])
    up, ug, out = set(range(len(preds))), set(range(len(gts))), []
    for s, i, j in pairs:
        if i in up and j in ug: out.append((i, j, s)); up.discard(i); ug.discard(j)
    return out, len(up), len(ug)

def inst_greedy_match(preds, gts, criterion="dice"):
    return greedy_match(preds, gts, inst_dice if criterion == "dice" else inst_iou)

def p1_image(preds, gts):
    """Official-style ARCADE F1 for one image: matched pairs score their Dice, every unmatched prediction or lesion scores 0."""
    if not preds and not gts: return 1.0
    pairs, n_up, n_ug = inst_greedy_match(preds, gts, "dice")
    return float(np.mean([s for _, _, s in pairs] + [0.0] * (n_up + n_ug)))

def full_metrics(preds_by_img, items):
    p1, dimg, iimg = [], [], []
    TP = FP = FN = pTP = pFP = pFN = 0
    for k, it in enumerate(items):
        preds, gts = preds_by_img[k], it["gts"]
        p1.append(p1_image(preds, gts))
        pairs, n_up, n_ug = inst_greedy_match(preds, gts, "iou")
        tp = sum(1 for _, _, s in pairs if s >= 0.5)
        TP += tp; FP += len(preds) - tp; FN += len(gts) - tp
        pr, gt = paint(preds, it["shape"]), paint(gts, it["shape"])
        a, b, c = np.count_nonzero(pr & gt), np.count_nonzero(pr & ~gt), np.count_nonzero(~pr & gt)
        pTP += a; pFP += b; pFN += c
        dimg.append(1.0 if a + b + c == 0 else 2 * a / (2 * a + b + c))
        iimg.append(1.0 if a + b + c == 0 else a / (a + b + c))
    P, R = TP / max(TP + FP, 1e-9), TP / max(TP + FN, 1e-9)
    return {"P1": float(np.mean(p1)), "F1": 2 * P * R / max(P + R, 1e-9), "Precision": P, "Recall": R, "TP": TP, "FP": FP, "FN": FN,
            "Dice": 2 * pTP / max(2 * pTP + pFP + pFN, 1e-9), "IoU": pTP / max(pTP + pFP + pFN, 1e-9),
            "Dice_img": float(np.mean(dimg)), "Dice_img_std": float(np.std(dimg)), "IoU_img": float(np.mean(iimg)),
            "pred_per_img": (TP + FP) / len(items), "per_image_P1": np.array(p1), "per_image_Dice": np.array(dimg)}

_g = [Inst(0, 0, np.ones((5, 5), bool))]
assert p1_image(_g, _g) == 1.0 and abs(p1_image(_g + [Inst(9, 9, np.ones((3, 3), bool))], _g) - 0.5) < 1e-9
print("self-check official-style F1: perfect = 1.0 | one extra false positive = 0.5  -> OK")

def polys_to_mask(segs, h, w):
    full = np.zeros((h, w), np.uint8)
    for poly in segs or []:
        if isinstance(poly, list) and len(poly) >= 6:
            cv2.fillPoly(full, [np.round(np.array(poly, np.float32).reshape(-1, 2)).astype(np.int32)], 1)
    return full

def load_split(split, with_images=True):
    coco = json.load(open(DATASET_ROOT / split / "annotations" / f"{split}.json"))
    anns = {}
    for a in coco["annotations"]:
        if a["category_id"] == STENOSIS_CATEGORY_ID: anns.setdefault(a["image_id"], []).append(a)
    imgs, masks, items = [], [], []
    for im in tqdm(sorted(coco["images"], key=lambda d: d["id"]), desc=f"ARCADE {split}", mininterval=5):
        h, w = im["height"], im["width"]
        if with_images:
            g = cv2.imread(str(DATASET_ROOT / split / "images" / im["file_name"]), cv2.IMREAD_GRAYSCALE)
            assert g.shape == (IMG_SIZE, IMG_SIZE), f"{im['file_name']} is {g.shape}, expected 512x512 (ARCADE native)"
            imgs.append(g)
        gts, union = [], np.zeros((h, w), np.uint8)
        for a in anns.get(im["id"], []):
            m = polys_to_mask(a.get("segmentation"), h, w); union |= m
            gi = inst_from_full(m.astype(bool))
            if gi is not None: gts.append(gi)
        masks.append(union); items.append({"file": im["file_name"], "shape": (h, w), "gts": gts})
    return (np.stack(imgs) if with_images else None), np.stack(masks), items

TRAIN_IMGS, TRAIN_MASKS, TRAIN_ITEMS = load_split("train")
VAL_IMGS, VAL_MASKS, VAL_ITEMS = load_split("val")
TEST_IMGS, TEST_MASKS, TEST_ITEMS = load_split("test")
MIN_AREA_REF = int(np.percentile([g.area for it in TRAIN_ITEMS + VAL_ITEMS for g in it["gts"]], 5))   # same definition as the VG-DeepLab notebook (train+val)
# the colour read used by the original U-Net / DeepLabV3+ / YOLO notebooks must equal 3 x the grey read used here
_f = DATASET_ROOT / "test" / "images" / TEST_ITEMS[0]["file"]; _c = cv2.imread(str(_f), cv2.IMREAD_COLOR)
GRAY_OK = bool((_c[..., 0] == _c[..., 1]).all() and (_c[..., 1] == _c[..., 2]).all() and (_c[..., 0] == TEST_IMGS[0]).all())
print("images are pure grey (colour read == grey read):", GRAY_OK)
if SMOKE_TEST:
    rs = np.random.default_rng(0)
    vi = np.sort(rs.choice(len(VAL_IMGS), min(40, len(VAL_IMGS)), replace=False)); ti = np.sort(rs.choice(len(TEST_IMGS), min(40, len(TEST_IMGS)), replace=False))
    VAL_IMGS, VAL_MASKS, VAL_ITEMS = VAL_IMGS[vi], VAL_MASKS[vi], [VAL_ITEMS[i] for i in vi]
    TEST_IMGS, TEST_MASKS, TEST_ITEMS = TEST_IMGS[ti], TEST_MASKS[ti], [TEST_ITEMS[i] for i in ti]
GTU = {"val": VAL_MASKS > 0, "test": TEST_MASKS > 0}; IMGS = {"val": VAL_IMGS, "test": TEST_IMGS}; ITEMS = {"val": VAL_ITEMS, "test": TEST_ITEMS}
N_TEST = len(TEST_ITEMS)
for s in ["val", "test"]:
    n = sum(len(i["gts"]) for i in ITEMS[s]); print(f"{s:5s}: {len(ITEMS[s])} images | {n} lesions | {n / len(ITEMS[s]):.2f} per image | images without a lesion: {sum(1 for i in ITEMS[s] if not i['gts'])}")
print("5th-percentile lesion area (train+val) =", MIN_AREA_REF, "px")

ARCADE root: /kaggle/input/datasets/nirmalgaud/arcade-dataset/arcade/stenosis
self-check official-style F1: perfect = 1.0 | one extra false positive = 0.5  -> OK


ARCADE train:   0%|          | 0/1000 [00:00<?, ?it/s]

ARCADE val:   0%|          | 0/200 [00:00<?, ?it/s]

ARCADE test:   0%|          | 0/300 [00:00<?, ?it/s]

images are pure grey (colour read == grey read): True
val  : 200 images | 406 lesions | 2.03 per image | images without a lesion: 0
test : 300 images | 386 lesions | 1.29 per image | images without a lesion: 0
5th-percentile lesion area (train+val) = 331 px


In [4]:
# ============================================================
# Candidates, post-processing search, operating points (same code as the VG-DeepLab notebook)
# ============================================================
def cand_dict(preds, gts, gtu_k):
    preds = sorted(preds, key=lambda t: -t.score)
    D = np.zeros((len(preds), len(gts)), np.float32)
    for i, pr in enumerate(preds):
        for j, g in enumerate(gts): D[i, j] = inst_dice(pr, g)
    inter = np.array([np.count_nonzero(pr.m & gtu_k[pr.y0:pr.y1, pr.x0:pr.x1]) for pr in preds], np.int64)
    return dict(preds=preds, scores=np.array([pr.score for pr in preds]), area=np.array([pr.area for pr in preds], np.int64),
                inter=inter, D=D, gt_area=int(gtu_k.sum()), ngt=len(gts))

def build_cands(probs, gtu, items, thr, min_area):
    """probs: (N,H,W) float; one dict per image with connected components (score = mean probability) sorted by score."""
    out = []
    for k in range(len(probs)):
        p = probs[k].astype(np.float32)
        n, lab, stats, _ = cv2.connectedComponentsWithStats((p > thr).astype(np.uint8), connectivity=8)
        preds = []
        for i in range(1, n):
            x, y, w, h, a = stats[i]
            if a < min_area: continue
            m = lab[y:y + h, x:x + w] == i
            preds.append(Inst(y, x, m, score=float(p[y:y + h, x:x + w][m].mean())))
        out.append(cand_dict(preds, items[k]["gts"], gtu[k]))
    return out

def keep_idx(sc, tau, rel, max_k, top1):
    n = len(sc)
    keep = [i for i in range(n) if sc[i] >= tau and sc[i] >= rel * sc[0]][:max_k]
    if not keep and top1 and n: keep = [0]
    return keep

def p1_from_D(Dsub, n_gt):
    n_pred = Dsub.shape[0]
    if n_pred == 0 and n_gt == 0: return 1.0
    scores, matched_p, matched_g = [], set(), set()
    if n_pred and n_gt:
        ii, jj = np.nonzero(Dsub > 0)
        for o in np.argsort(-Dsub[ii, jj], kind="stable"):
            i, j = int(ii[o]), int(jj[o])
            if i not in matched_p and j not in matched_g:
                scores.append(float(Dsub[i, j])); matched_p.add(i); matched_g.add(j)
    scores += [0.0] * ((n_pred - len(matched_p)) + (n_gt - len(matched_g)))
    return float(np.mean(scores))

def eval_cfg(C, tau, rel, max_k, top1):
    d_sum = p_sum = 0.0
    for c in C:
        keep = keep_idx(c["scores"], tau, rel, max_k, top1)
        P = int(c["area"][keep].sum()) if keep else 0; I = int(c["inter"][keep].sum()) if keep else 0
        d_sum += 1.0 if P + c["gt_area"] == 0 else 2.0 * I / (P + c["gt_area"])
        p_sum += p1_from_D(c["D"][keep] if keep else np.zeros((0, c["ngt"]), np.float32), c["ngt"])
    return d_sum / len(C), p_sum / len(C)

def select_insts(c, cfg):
    return [c["preds"][i] for i in keep_idx(c["scores"], cfg["tau"], cfg["rel"], cfg["max_k"], cfg["top1"])]

GRID_SEM = dict(thr=[0.30, 0.35, 0.40, 0.45, 0.50, 0.55], min_area=sorted({150, MIN_AREA_REF}),
                tau=[0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8], rel=[0.0, 0.5, 0.75], max_k=[1, 2, 3, 99], top1=[True, False])
GRID_YOLO = dict(thr=[0.0], min_area=[0], tau=[0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8], rel=[0.0, 0.5, 0.75], max_k=[1, 2, 3, 99], top1=[True, False])

def search_postproc(cands_fn, grid):
    """cands_fn(thr, min_area) -> candidate dicts of the VALIDATION images."""
    rows = []
    for thr, ma in itertools.product(grid["thr"], grid["min_area"]):
        C = cands_fn(thr, ma)
        for tau, rel, mk, t1 in itertools.product(grid["tau"], grid["rel"], grid["max_k"], grid["top1"]):
            d, p = eval_cfg(C, tau, rel, mk, t1)
            rows.append(dict(thr=thr, min_area=ma, tau=tau, rel=rel, max_k=mk, top1=t1, dice_img=d, p1=p))
    return pd.DataFrame(rows)

def pick_ops(df):
    ops = {}
    for tag, sub in [("ARCADE", df), ("DEPLOY", df[~df.top1])]:
        for obj, key in [("maxDice", df.dice_img), ("maxF1", df.p1), ("balanced", (df.dice_img + df.p1) / 2)]:
            r = df.loc[key.loc[sub.index].idxmax()]
            ops[f"{obj}-{tag}"] = dict(thr=float(r.thr), min_area=int(r.min_area), tau=float(r.tau), rel=float(r.rel),
                                       max_k=int(r.max_k), top1=bool(r.top1), val_dice=float(r.dice_img), val_p1=float(r.p1))
    return ops

def pixel_confusion(preds_by_img, items):
    TP = FP = FN = total = 0
    for preds, it in zip(preds_by_img, items):
        h, w = it["shape"]; total += h * w
        pr, gt = paint(preds, it["shape"]), paint(it["gts"], it["shape"])
        TP += int(np.count_nonzero(pr & gt)); FP += int(np.count_nonzero(pr & ~gt)); FN += int(np.count_nonzero(~pr & gt))
    TN = total - TP - FP - FN
    return dict(TP=TP, FP=FP, FN=FN, TN=TN, precision=TP / max(TP + FP, 1), recall=TP / max(TP + FN, 1), specificity=TN / max(TN + FP, 1),
                accuracy=(TP + TN) / max(total, 1), dice=2 * TP / max(2 * TP + FP + FN, 1), iou=TP / max(TP + FP + FN, 1))

def match_counts(preds_by_img, items, thr, fn=inst_iou):
    """Per image: (true positives at overlap >= thr, number of predictions, number of GT lesions)."""
    tp, npred, ngt = [], [], []
    for preds, it in zip(preds_by_img, items):
        pairs, _, _ = greedy_match(preds, it["gts"], fn)
        tp.append(sum(1 for _, _, s in pairs if s >= thr)); npred.append(len(preds)); ngt.append(len(it["gts"]))
    return np.array(tp), np.array(npred), np.array(ngt)

def f1_from_counts(cnt):
    tp, npred, ngt = cnt
    return 2 * tp.sum() / max(npred.sum() + ngt.sum(), 1)

def cap_cands(C, max_per_img):
    out = []
    for c in C:
        n = min(len(c["preds"]), max_per_img)
        out.append(dict(preds=c["preds"][:n], scores=c["scores"][:n], area=c["area"][:n], inter=c["inter"][:n], D=c["D"][:n], gt_area=c["gt_area"], ngt=c["ngt"]))
    return out

def map_prep(C, items):
    ious = []
    for c, it in zip(C, items):
        m = np.zeros((len(c["preds"]), len(it["gts"])), np.float32)
        for i, p in enumerate(c["preds"]):
            for j, g in enumerate(it["gts"]): m[i, j] = inst_iou(p, g)
        ious.append(m)
    return ious, np.array([len(it["gts"]) for it in items]), [np.asarray(c["scores"], np.float64) for c in C]

def map_from_prep(prep, idx=None, thrs=np.arange(0.50, 0.96, 0.05)):
    """COCO-style mask AP (101-point). idx = optional bootstrap resample of images. Returns (AP50, AP50-95)."""
    ious, ngt, scores = prep
    if idx is None: idx = np.arange(len(ngt))
    n_gt = int(ngt[idx].sum())
    order = sorted(((float(scores[k][i]), j, i) for j, k in enumerate(idx) for i in range(len(scores[k]))), key=lambda d: -d[0])
    aps = []
    for thr in thrs:
        used = [np.zeros(ngt[k], bool) for k in idx]; tp = np.zeros(len(order))
        for n, (_, j, i) in enumerate(order):
            k = idx[j]
            if ngt[k] == 0: continue
            cand = np.where(~used[j], ious[k][i], -1.0); g = int(cand.argmax())
            if cand[g] >= thr: used[j][g] = True; tp[n] = 1
        ctp, cfp = np.cumsum(tp), np.cumsum(1 - tp); rec = ctp / max(n_gt, 1)
        prec = np.maximum.accumulate((ctp / np.maximum(ctp + cfp, 1e-9))[::-1])[::-1] if len(order) else np.zeros(0)
        aps.append(float(np.mean([prec[rec >= r].max() if np.any(rec >= r) else 0.0 for r in np.linspace(0, 1, 101)])))
    return aps[0], float(np.mean(aps))

# ---- bootstrap over test images (the same resamples for every model -> paired comparisons) ----
BI = np.random.default_rng(0).integers(0, N_TEST, (N_BOOT, N_TEST))
BI_MAP = BI[:N_BOOT_MAP]
def boot_mean(v): return v[BI].mean(1)
def boot_f1(cnt):
    tp, npred, ngt = cnt
    return 2 * tp[BI].sum(1) / np.maximum(npred[BI].sum(1) + ngt[BI].sum(1), 1)
def ci95(b): return np.percentile(b, [2.5, 97.5])
def fmt_ci(v, b): lo, hi = ci95(b); return f"{v:.4f} [{lo:.3f},{hi:.3f}]"
print("metric code ready | bootstrap resamples:", N_BOOT, "(means, F1) /", N_BOOT_MAP, "(mAP)")

metric code ready | bootstrap resamples: 10000 (means, F1) / 200 (mAP)


In [5]:
class VGNet(nn.Module):
    """5-fold models (VG-DeepLab notebook). aux: vessel channel. refine: vessel-guided refinement head. forward -> (stenosis logit, coarse logit, vessel logit or None)."""
    def __init__(self, aux=True, refine=True):
        super().__init__()
        self.aux, self.refine = aux, refine
        self.net = smp.DeepLabV3Plus(encoder_name="resnet101", encoder_weights=None, in_channels=3, classes=2 if aux else 1)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std",  torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
        if refine:
            cin = 3 if aux else 2
            self.ref = nn.Sequential(nn.Conv2d(cin, 24, 3, padding=1), nn.ReLU(inplace=True),
                                     nn.Conv2d(24, 24, 3, padding=2, dilation=2), nn.ReLU(inplace=True),
                                     nn.Conv2d(24, 24, 3, padding=4, dilation=4), nn.ReLU(inplace=True),
                                     nn.Conv2d(24, 24, 3, padding=1), nn.ReLU(inplace=True), nn.Conv2d(24, 1, 1))
    def forward(self, x):
        out = self.net((x - self.mean) / self.std)
        s = out[:, :1]; v = out[:, 1:2] if self.aux else None
        if not self.refine: return s, s, v
        f = [x[:, :1] * 2 - 1, torch.sigmoid(s.detach()) * 2 - 1]
        if self.aux: f.append(torch.sigmoid(v.detach()) * 2 - 1)
        return s + self.ref(torch.cat(f, 1)), s, v

def n_params(m): return sum(p.numel() for p in m.parameters()) / 1e6

def to_x(imgs):
    return torch.from_numpy(imgs).float().div(255).unsqueeze(1).repeat(1, 3, 1, 1).to(DEVICE)      # grey image in 3 identical channels, [0,1]


@torch.no_grad()
def predict_ms(model, imgs, bs=8):
    """5-fold models: test-time augmentation 'ms' (2 scales 1.0 / 1.25 x horizontal flip), exactly as in the VG-DeepLab notebook."""
    model.eval(); S = []
    for st in range(0, len(imgs), bs):
        x = to_x(imgs[st:st + bs]); acc = 0.0
        for sc in (1.0, 1.25):
            xs = x if sc == 1.0 else F.interpolate(x, scale_factor=sc, mode="bilinear", align_corners=False)
            for flip in (False, True):
                xi = torch.flip(xs, dims=[3]) if flip else xs
                with autocast("cuda", enabled=USE_CUDA): s2, _, _ = model(xi)
                o = torch.sigmoid(s2.float())
                if flip: o = torch.flip(o, dims=[3])
                if sc != 1.0: o = F.interpolate(o, size=x.shape[-2:], mode="bilinear", align_corners=False)
                acc = acc + o
        S.append((acc / 4)[:, 0].cpu().numpy().astype(np.float16))
    return np.concatenate(S)


In [6]:
# ============================================================
# Pool (train + val = 1200 images), the 5 folds, and the fold weights
# ============================================================
from sklearn.model_selection import StratifiedKFold
POOL_IMGS = np.concatenate([TRAIN_IMGS, VAL_IMGS]); POOL_MASKS = np.concatenate([TRAIN_MASKS, VAL_MASKS]); POOL_ITEMS = TRAIN_ITEMS + VAL_ITEMS
GTU_POOL = POOL_MASKS > 0; N_POOL = len(POOL_ITEMS)
assert N_POOL == 1200, f"expected 1200 pooled images, got {N_POOL}"
y_strat = np.array([min(len(it["gts"]), 3) for it in POOL_ITEMS])
FOLDS = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(np.zeros(N_POOL), y_strat))
print("fold sizes (held-out images):", [len(va) for _, va in FOLDS], "| lesions in pool:", sum(len(it["gts"]) for it in POOL_ITEMS))

def find_fold_ckpts(prefix):
    found = {}
    for root in [Path("/kaggle/input"), Path("/content"), BASE]:
        if root.exists():
            for p in root.rglob(f"{prefix}_f[0-4].pt"):
                f = int(p.stem.split("_f")[-1]); found.setdefault(f, p)
    assert sorted(found) == [0, 1, 2, 3, 4], f"{prefix}_f0..4.pt not all found (found folds: {sorted(found)}). Add the output of the first VG-DeepLab notebook as an input."
    return found
CK = {"B": find_fold_ckpts("dl3_r101"), "N": find_fold_ckpts("vg_dl3")}
for k, v in CK.items(): print(k, {f: str(p) for f, p in v.items()})


fold sizes (held-out images): [240, 240, 240, 240, 240] | lesions in pool: 2031
B {0: '/kaggle/input/notebooks/fahimmuntasir1168/arcade-stenosis-vg-deeplab/run/dl3_r101_f0.pt', 1: '/kaggle/input/notebooks/fahimmuntasir1168/arcade-stenosis-vg-deeplab/run/dl3_r101_f1.pt', 2: '/kaggle/input/notebooks/fahimmuntasir1168/arcade-stenosis-vg-deeplab/run/dl3_r101_f2.pt', 3: '/kaggle/input/notebooks/fahimmuntasir1168/arcade-stenosis-vg-deeplab/run/dl3_r101_f3.pt', 4: '/kaggle/input/notebooks/fahimmuntasir1168/arcade-stenosis-vg-deeplab/run/dl3_r101_f4.pt'}
N {0: '/kaggle/input/notebooks/fahimmuntasir1168/arcade-stenosis-vg-deeplab/run/vg_dl3_f0.pt', 1: '/kaggle/input/notebooks/fahimmuntasir1168/arcade-stenosis-vg-deeplab/run/vg_dl3_f1.pt', 2: '/kaggle/input/notebooks/fahimmuntasir1168/arcade-stenosis-vg-deeplab/run/vg_dl3_f2.pt', 3: '/kaggle/input/notebooks/fahimmuntasir1168/arcade-stenosis-vg-deeplab/run/vg_dl3_f3.pt', 4: '/kaggle/input/notebooks/fahimmuntasir1168/arcade-stenosis-vg-deeplab/run

In [7]:
# ============================================================
# Probability maps: OOF (each pooled image by the fold model that did not train on it) and test (mean of the 5 fold models)
# ============================================================
def oof_and_test(key, aux, refine):
    oof = np.zeros((N_POOL, IMG_SIZE, IMG_SIZE), np.float16); acc = np.zeros(TEST_IMGS.shape, np.float32); t0 = time.time()
    for f, (tr, va) in enumerate(FOLDS):
        net = VGNet(aux, refine); net.load_state_dict(torch.load(CK[key][f], map_location="cpu", weights_only=True), strict=True); net.to(DEVICE).eval()
        oof[va] = predict_ms(net, POOL_IMGS[va]); acc += predict_ms(net, TEST_IMGS).astype(np.float32); del net
        if USE_CUDA: torch.cuda.empty_cache()
        print(f"{key}: fold {f + 1}/5 done | {time.time() - t0:.0f}s")
    return oof, (acc / 5).astype(np.float16)
OOF, TEST = {}, {}
OOF["B"], TEST["B"] = oof_and_test("B", False, False)
OOF["N"], TEST["N"] = oof_and_test("N", True, True)
GTU_TEST = TEST_MASKS > 0
print("maps ready")

B: fold 1/5 done | 53s
B: fold 2/5 done | 103s
B: fold 3/5 done | 154s
B: fold 4/5 done | 205s
B: fold 5/5 done | 257s
N: fold 1/5 done | 64s
N: fold 2/5 done | 127s
N: fold 3/5 done | 190s
N: fold 4/5 done | 252s
N: fold 5/5 done | 315s
maps ready


In [8]:
# ============================================================
# True F1 per image, scoring helper, and a reproduction check against the thesis
# ============================================================
def truef1_image(preds, gts):
    """Usual F1 form with Dice credit: 2 * sum(matched Dice) / (2M + unmatched predictions + unmatched lesions)."""
    if not preds and not gts: return 1.0
    pairs, n_up, n_ug = inst_greedy_match(preds, gts, "dice")
    return 2.0 * sum(s for _, _, s in pairs) / (2 * len(pairs) + n_up + n_ug)
_g = [Inst(0, 0, np.ones((5, 5), bool))]
assert truef1_image(_g, _g) == 1.0 and abs(truef1_image(_g + [Inst(9, 9, np.ones((3, 3), bool))], _g) - 2 / 3) < 1e-9
print("self-check true F1: perfect = 1.0 | one extra false positive = 0.667 (official-style gives 0.5) -> OK")

def score(probs, gtu, items, op):
    C = build_cands(probs, gtu, items, op["thr"], op["min_area"]); preds = [select_insts(c, op) for c in C]
    fm = full_metrics(preds, items)
    return dict(preds=preds, dice=fm["per_image_Dice"], p1=fm["per_image_P1"], tf1=np.array([truef1_image(p, it["gts"]) for p, it in zip(preds, items)]),
                cnt=match_counts(preds, items, 0.5), npred=np.array([len(p) for p in preds]))

DATA = {"test": (TEST, GTU_TEST, TEST_ITEMS), "oof": (OOF, GTU_POOL, POOL_ITEMS)}
OWN = {}
for split, (maps, gtu, items) in DATA.items():
    for key in ["B", "N"]:
        for opn in ["balanced-DEPLOY", "maxF1-ARCADE"]:
            OWN[(split, key, opn)] = score(maps[key], gtu, items, FROZEN_OPS[key][opn])
EXPECTED = {("test", "B", "balanced-DEPLOY"): (0.6197, 0.5165), ("test", "N", "balanced-DEPLOY"): (0.6358, 0.5436),
            ("test", "B", "maxF1-ARCADE"): (None, 0.5230), ("test", "N", "maxF1-ARCADE"): (None, 0.5579),
            ("oof", "B", "balanced-DEPLOY"): (0.5941, 0.4653), ("oof", "N", "balanced-DEPLOY"): (0.6028, 0.4631)}
rows = []
for k, (ed, ep) in EXPECTED.items():
    s = OWN[k]
    for m, e, got in [("Dice/img", ed, s["dice"].mean()), ("official-style F1", ep, s["p1"].mean())]:
        if e is None: continue
        rows.append(dict(split=k[0], model=k[1], op=k[2], metric=m, thesis=e, here=round(float(got), 4), diff=round(float(got - e), 4), status="OK" if abs(got - e) <= 0.005 else "MISMATCH"))
REPRO = pd.DataFrame(rows); REPRO.to_csv(OUT_DIR / "reproduction_check.csv", index=False); print(REPRO.to_string(index=False))
if (REPRO.status == "MISMATCH").any(): print("\nWARNING: a number does not reproduce - send the output before using anything below.")

self-check true F1: perfect = 1.0 | one extra false positive = 0.667 (official-style gives 0.5) -> OK
split model              op            metric  thesis   here  diff status
 test     B balanced-DEPLOY          Dice/img  0.6197 0.6197   0.0     OK
 test     B balanced-DEPLOY official-style F1  0.5165 0.5165  -0.0     OK
 test     N balanced-DEPLOY          Dice/img  0.6358 0.6358  -0.0     OK
 test     N balanced-DEPLOY official-style F1  0.5436 0.5436  -0.0     OK
 test     B    maxF1-ARCADE official-style F1  0.5230 0.5230   0.0     OK
 test     N    maxF1-ARCADE official-style F1  0.5579 0.5579  -0.0     OK
  oof     B balanced-DEPLOY          Dice/img  0.5941 0.5941  -0.0     OK
  oof     B balanced-DEPLOY official-style F1  0.4653 0.4653   0.0     OK
  oof     N balanced-DEPLOY          Dice/img  0.6028 0.6028  -0.0     OK
  oof     N balanced-DEPLOY official-style F1  0.4631 0.4631  -0.0     OK


In [9]:
# ============================================================
# 1) True F1 next to the official-style F1 (own frozen operating points)
# ============================================================
BI_T = np.random.default_rng(0).integers(0, len(TEST_ITEMS), (N_BOOT, len(TEST_ITEMS)))
BI_O = np.random.default_rng(1).integers(0, N_POOL, (N_BOOT, N_POOL))
BIX = {"test": BI_T, "oof": BI_O}
def iv(b, a=0.05): return np.percentile(b, [100 * a / 2, 100 * (1 - a / 2)])
rows = []
for (split, key, opn), s in OWN.items():
    bi = BIX[split]
    for m in ["p1", "tf1"]:
        v = s[m]; lo, hi = iv(v[bi].mean(1))
        rows.append(dict(split=split, model=key, operating_point=opn, metric={"p1": "official-style F1", "tf1": "true F1 (Dice credit)"}[m],
                         value=round(float(v.mean()), 4), ci95=f"[{lo:.3f}, {hi:.3f}]"))
    rows.append(dict(split=split, model=key, operating_point=opn, metric="images with exactly 1 prediction and 1 lesion",
                     value=int(((s["npred"] == 1) & (np.array([len(it["gts"]) for it in DATA[split][2]]) == 1)).sum()), ci95=""))
T1 = pd.DataFrame(rows); T1.to_csv(OUT_DIR / "T1_true_f1.csv", index=False); print(T1.to_string(index=False))

split model operating_point                                        metric    value           ci95
 test     B balanced-DEPLOY                             official-style F1   0.5165 [0.489, 0.544]
 test     B balanced-DEPLOY                         true F1 (Dice credit)   0.5587 [0.535, 0.583]
 test     B balanced-DEPLOY images with exactly 1 prediction and 1 lesion 159.0000               
 test     B    maxF1-ARCADE                             official-style F1   0.5230 [0.496, 0.551]
 test     B    maxF1-ARCADE                         true F1 (Dice credit)   0.5621 [0.538, 0.586]
 test     B    maxF1-ARCADE images with exactly 1 prediction and 1 lesion 170.0000               
 test     N balanced-DEPLOY                             official-style F1   0.5436 [0.515, 0.572]
 test     N balanced-DEPLOY                         true F1 (Dice credit)   0.5798 [0.555, 0.605]
 test     N balanced-DEPLOY images with exactly 1 prediction and 1 lesion 167.0000               
 test     N    maxF1

In [10]:
# ============================================================
# 2) Same operating point for B and N
#    (a) B's frozen deploy values for both, (b) N's frozen deploy values for both,
#    (c) one joint deploy setting chosen on the 1200 OOF images: max of the mean over B and N of (Dice/img + official F1) / 2
# ============================================================
t0 = time.time(); G = {}
for key in ["B", "N"]:
    G[key] = search_postproc(lambda thr, ma, key=key: build_cands(OOF[key], GTU_POOL, POOL_ITEMS, thr, ma), GRID_SEM)
    print(key, "OOF grid done", len(G[key]), "settings", f"{time.time() - t0:.0f}s")
cols = ["thr", "min_area", "tau", "rel", "max_k", "top1"]
J = G["B"].merge(G["N"], on=cols, suffixes=("_B", "_N")); J = J[~J.top1]
J["joint"] = ((J.dice_img_B + J.p1_B) / 2 + (J.dice_img_N + J.p1_N) / 2) / 2
r = J.loc[J.joint.idxmax()]
JOINT = dict(thr=float(r.thr), min_area=int(r.min_area), tau=float(r.tau), rel=float(r.rel), max_k=int(r.max_k), top1=False)
print("joint OOF setting:", JOINT); J.sort_values("joint", ascending=False).head(20).to_csv(OUT_DIR / "joint_setting_top20_oof.csv", index=False)
COMMON = {"B's values for both": FROZEN_OPS["B"][HEADLINE], "N's values for both": FROZEN_OPS["N"][HEADLINE], "joint OOF setting": JOINT}

def paired(sB, sN, bi, alphas=(0.05, ALPHA_BONF)):
    out = {}
    for m, name in [("dice", "Dice/img"), ("p1", "official-style F1"), ("tf1", "true F1")]:
        d = sN[m].mean() - sB[m].mean(); b = sN[m][bi].mean(1) - sB[m][bi].mean(1); out[name] = (d, b, sB[m].mean(), sN[m].mean())
    def f1b(c): tp, npd, ng = c; return 2 * tp[bi].sum(1) / np.maximum(npd[bi].sum(1) + ng[bi].sum(1), 1)
    def f1(c): tp, npd, ng = c; return 2 * tp.sum() / max(npd.sum() + ng.sum(), 1)
    out["instance F1 @ IoU 0.5"] = (f1(sN["cnt"]) - f1(sB["cnt"]), f1b(sN["cnt"]) - f1b(sB["cnt"]), f1(sB["cnt"]), f1(sN["cnt"]))
    rows = []
    for name, (d, b, vB, vN) in out.items():
        l95, h95 = iv(b, 0.05); lb, hb = iv(b, ALPHA_BONF)
        rows.append(dict(metric=name, first=round(float(vB), 4), second=round(float(vN), 4), N_minus_B=round(float(d), 4), ci95=f"[{l95:+.4f}, {h95:+.4f}]", ci9967=f"[{lb:+.4f}, {hb:+.4f}]",
                         excl0_95=bool(l95 > 0 or h95 < 0), excl0_9967=bool(lb > 0 or hb < 0)))
    return rows
rows = []
for cname, op in COMMON.items():
    for split, (maps, gtu, items) in DATA.items():
        sB, sN = score(maps["B"], gtu, items, op), score(maps["N"], gtu, items, op)
        for rr in paired(sB, sN, BIX[split]):
            rows.append(dict(setting=cname, split=split, **rr))
T2 = pd.DataFrame(rows); T2.to_csv(OUT_DIR / "T2_common_operating_point.csv", index=False); print(T2.to_string(index=False))
print("(mAP does not depend on the operating point: it ranks all candidates, so it is the same as in Table 4.5)")

B OOF grid done 2016 settings 129s
N OOF grid done 2016 settings 256s
joint OOF setting: {'thr': 0.4, 'min_area': 150, 'tau': 0.2, 'rel': 0.75, 'max_k': 99, 'top1': False}
            setting split                metric  first  second  N_minus_B               ci95             ci9967  excl0_95  excl0_9967
B's values for both  test              Dice/img 0.6197  0.6373     0.0176 [+0.0098, +0.0256] [+0.0059, +0.0294]      True        True
B's values for both  test     official-style F1 0.5165  0.5387     0.0222 [+0.0078, +0.0367] [+0.0009, +0.0445]      True        True
B's values for both  test               true F1 0.5587  0.5783     0.0196 [+0.0089, +0.0304] [+0.0034, +0.0359]      True        True
B's values for both  test instance F1 @ IoU 0.5 0.4591  0.4963     0.0372 [+0.0085, +0.0662] [-0.0070, +0.0804]      True       False
B's values for both   oof              Dice/img 0.5941  0.5985     0.0044 [-0.0036, +0.0127] [-0.0080, +0.0162]     False       False
B's values for both   oo

In [11]:
# ============================================================
# 3) Bonferroni: N - B at the thesis operating points, 95 % and 99.67 %, 10,000 resamples for every metric (mAP included)
# ============================================================
rows = []
for split, (maps, gtu, items) in DATA.items():
    sB, sN = OWN[(split, "B", HEADLINE)], OWN[(split, "N", HEADLINE)]
    rows += [dict(split=split, **rr) for rr in paired(sB, sN, BIX[split])]
    t0 = time.time(); bi = BIX[split][:N_BOOT_MAP_BIG]; prep = {}
    for key in ["B", "N"]:
        Cm = cap_cands(build_cands(maps[key], gtu, items, CAND_THR, CAND_MIN_AREA), CAND_MAX); prep[key] = map_prep(Cm, items)
    full = {k: map_from_prep(prep[k]) for k in prep}
    mb = {k: np.array([map_from_prep(prep[k], idx) for idx in tqdm(bi, desc=f"mAP {split} {k}", mininterval=30)]) for k in prep}
    for j, name in [(0, "mAP50"), (1, "mAP50-95")]:
        d = full["N"][j] - full["B"][j]; b = mb["N"][:, j] - mb["B"][:, j]; l95, h95 = iv(b, 0.05); lb, hb = iv(b, ALPHA_BONF)
        rows.append(dict(split=split, metric=name, first=round(float(full["B"][j]), 4), second=round(float(full["N"][j]), 4), N_minus_B=round(float(d), 4), ci95=f"[{l95:+.4f}, {h95:+.4f}]", ci9967=f"[{lb:+.4f}, {hb:+.4f}]",
                         excl0_95=bool(l95 > 0 or h95 < 0), excl0_9967=bool(lb > 0 or hb < 0)))
    print(split, f"mAP bootstrap {time.time() - t0:.0f}s")
T3 = pd.DataFrame(rows); T3.to_csv(OUT_DIR / "T3_bonferroni_N_vs_B.csv", index=False); print(T3.to_string(index=False))

mAP test B:   0%|          | 0/10000 [00:00<?, ?it/s]

mAP test N:   0%|          | 0/10000 [00:00<?, ?it/s]

test mAP bootstrap 726s


mAP oof B:   0%|          | 0/10000 [00:00<?, ?it/s]

mAP oof N:   0%|          | 0/10000 [00:00<?, ?it/s]

oof mAP bootstrap 2301s
split                metric  first  second  N_minus_B               ci95             ci9967  excl0_95  excl0_9967
 test              Dice/img 0.6197  0.6358     0.0161 [+0.0074, +0.0249] [+0.0032, +0.0293]      True        True
 test     official-style F1 0.5165  0.5436     0.0271 [+0.0128, +0.0421] [+0.0062, +0.0498]      True        True
 test               true F1 0.5587  0.5798     0.0212 [+0.0101, +0.0324] [+0.0045, +0.0384]      True        True
 test instance F1 @ IoU 0.5 0.4591  0.4864     0.0273 [-0.0023, +0.0574] [-0.0171, +0.0712]     False       False
 test                 mAP50 0.3126  0.3555     0.0428 [+0.0074, +0.0806] [-0.0088, +0.0991]      True       False
 test              mAP50-95 0.0798  0.0908     0.0110 [+0.0025, +0.0215] [-0.0018, +0.0258]      True       False
  oof              Dice/img 0.5941  0.6028     0.0087 [+0.0007, +0.0167] [-0.0035, +0.0205]      True       False
  oof     official-style F1 0.4653  0.4631    -0.0023 [-0.0121, 

In [12]:
# ============================================================
# 4) Optional: 99.67 % intervals for all 15 pairs of the six models, from unified_test_predictions.npz (deploy setting)
#    Instances = 8-connected components of each saved mask (for the semantic models this is exactly the kept set;
#    for YOLOv8s-seg touching masks merge, so its row is checked against Table 4.1 below).
# ============================================================
NPZ = next((p for root in [Path("/kaggle/input"), BASE] if root.exists() for p in root.rglob("unified_test_predictions.npz")), None)
if NPZ is None:
    print("unified_test_predictions.npz not attached -> section 4 skipped")
else:
    z = np.load(NPZ); files = list(z["files"]); order = [files.index(it["file"]) for it in TEST_ITEMS]
    names = [k for k in z.files if k != "files"]; S6 = {}
    for n in names:
        masks = np.unpackbits(z[n], axis=-1)[:, :, :IMG_SIZE].astype(bool)[order]; preds = []
        for m in masks:
            k_, lab = cv2.connectedComponents(m.astype(np.uint8), connectivity=8)
            preds.append([inst_from_full(lab == i) for i in range(1, k_)])
        fm = full_metrics(preds, TEST_ITEMS)
        S6[n] = dict(dice=fm["per_image_Dice"], p1=fm["per_image_P1"], tf1=np.array([truef1_image(p, it["gts"]) for p, it in zip(preds, TEST_ITEMS)]),
                     cnt=match_counts(preds, TEST_ITEMS, 0.5))
        print(f"{n:22s} Dice/img {fm['Dice_img']:.4f} | official F1 {fm['P1']:.4f} | true F1 {S6[n]['tf1'].mean():.4f}")
    print("Table 4.1 (thesis): U-Net 0.5604/0.4359, Att U-Net 0.5632/0.4290, YOLOv8s-seg 0.5354/0.2162, DeepLabV3+ 0.5858/0.4864, B 0.6197/0.5165, N 0.6358/0.5436")
    rows = []
    for a, b in itertools.combinations(names, 2):
        for rr in paired(S6[a], S6[b], BI_T): rows.append(dict(A=a, B_model=b, **rr))
    T4 = pd.DataFrame(rows).rename(columns={"N_minus_B": "second_minus_first"}); T4.to_csv(OUT_DIR / "T4_all_pairs_bonferroni.csv", index=False)
    print(T4.to_string(index=False))

U_Net_v2               Dice/img 0.5604 | official F1 0.4359 | true F1 0.4904
Attention_U_Net_v2     Dice/img 0.5632 | official F1 0.4290 | true F1 0.4849
YOLOv8s_seg            Dice/img 0.5354 | official F1 0.3311 | true F1 0.4092
DeepLabV3_R101         Dice/img 0.5858 | official F1 0.4864 | true F1 0.5281
B_DeepLabV3_x5         Dice/img 0.6197 | official F1 0.5165 | true F1 0.5587
N_VG_DeepLab_x5        Dice/img 0.6358 | official F1 0.5436 | true F1 0.5798
Table 4.1 (thesis): U-Net 0.5604/0.4359, Att U-Net 0.5632/0.4290, YOLOv8s-seg 0.5354/0.2162, DeepLabV3+ 0.5858/0.4864, B 0.6197/0.5165, N 0.6358/0.5436
                 A            B_model                metric  first  second  second_minus_first               ci95             ci9967  excl0_95  excl0_9967
          U_Net_v2 Attention_U_Net_v2              Dice/img 0.5604  0.5632              0.0028 [-0.0141, +0.0192] [-0.0227, +0.0275]     False       False
          U_Net_v2 Attention_U_Net_v2     official-style F1 0.4359  0.4290  

In [13]:
shutil.make_archive(str(BASE / "truef1_outputs"), "zip", OUT_DIR)
print("Errors:", ERRORS if ERRORS else "none")
print("Done. Download /kaggle/working/truef1_outputs.zip (Output tab) and send it to Claude.")

Errors: none
Done. Download /kaggle/working/truef1_outputs.zip (Output tab) and send it to Claude.
